In [1]:
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

print("Loading Multi-Agent System...")

# Agent 1 (RNN) - fraud score from transactions
agent1_rnn = tf.keras.models.load_model("../models/agent1_rnn.h5", compile=False)
print("✅ Agent 1 (RNN) loaded.")

# Agent 2 (Random Forest) - default score from applicant profile
with open("../models/agent2_rf.pkl", "rb") as f:
    agent2_rf = pickle.load(f)
print("✅ Agent 2 (Random Forest) loaded.")

print("\n🎉 Agents 1 and 2 are ready for Agent 3!")

Loading Multi-Agent System...
✅ Agent 1 (RNN) loaded.
✅ Agent 2 (Random Forest) loaded.

🎉 Agents 1 and 2 are ready for Agent 3!


In [3]:
RF_THRESHOLD = 0.5      # chosen from the threshold analysis (precision 0.17, recall 0.61)
FRAUD_THRESHOLD = 0.5   # RNN: precision 0.80, recall 0.83 at this threshold

def agent3_rules(customer_cc_data, customer_hc_data):
    """Takes the loan decision from the two agents' scores."""
    prob_fraud = agent1_rnn.predict(
        customer_cc_data.reshape(1, 1, -1), verbose=0)[0][0]
    prob_default = agent2_rf.predict_proba(
        customer_hc_data.reshape(1, -1))[0][1]

    if prob_fraud > FRAUD_THRESHOLD:
        decision, reason = "REJECTED ❌", "High fraud risk in transactions"
    elif prob_default > RF_THRESHOLD:
        decision, reason = "REJECTED ❌", "High default risk based on profile"
    else:
        decision, reason = "APPROVED ✅", "Low fraud risk and low default risk"

    return {
        "Agent 1 (RNN) Fraud Risk": f"{prob_fraud*100:.2f}%",
        "Agent 2 (RF) Default Risk": f"{prob_default*100:.2f}%",
        "Final Decision": decision,
        "Reason": reason,
    }

In [17]:
import requests

OLLAMA_MODEL = "llama3.2:1b"

# Feature names come from hc_data.pkl
with open("../data/clean/hc_data.pkl", "rb") as f:
    hc_feature_names = pickle.load(f)[4]

# The 3 most important profile features for the Random Forest
top_idx = np.argsort(agent2_rf.feature_importances_)[-3:][::-1]

def factor_sentence(name, v):
    """Plain-language meaning of one standardized feature (v = std vs average)."""
    if name.startswith("EXT_SOURCE"):
        label = name.replace("EXT_SOURCE_", "external credit score ")
        if v > 0.5:  return f"{label} is higher than average (favorable)"
        if v < -0.5: return f"{label} is lower than average (unfavorable)"
        return f"{label} is close to average"
    if name == "DAYS_EMPLOYED":
        # stored as negative days: more negative = employed for longer
        if v < -0.5: return "employment length is longer than average (favorable)"
        if v > 0.5:  return "employment length is shorter than average (unfavorable)"
        return "employment length is close to average"
    if v > 0.5:  return f"{name} is higher than average"
    if v < -0.5: return f"{name} is lower than average"
    return f"{name} is close to average"

def top_features_text(customer_hc_data):
    return "; ".join(
        factor_sentence(hc_feature_names[i], customer_hc_data[i]) for i in top_idx)

def explain_decision(result, customer_hc_data):
    """Template justification built only from the real facts (safe fallback)."""
    fraud = result["Agent 1 (RNN) Fraud Risk"]
    default = result["Agent 2 (RF) Default Risk"]
    reason = result["Reason"]
    factors = top_features_text(customer_hc_data)
    if reason.startswith("High fraud"):
        return (f"The application is rejected: the fraud risk of {fraud} is above the "
                f"{FRAUD_THRESHOLD*100:.0f}% limit. The default risk is {default}. "
                f"Profile factors: {factors}.")
    elif reason.startswith("High default"):
        return (f"The application is rejected: the default risk of {default} is above the "
                f"{RF_THRESHOLD*100:.0f}% limit. The fraud risk is low ({fraud}). "
                f"Profile factors: {factors}.")
    return (f"The application is approved: the fraud risk ({fraud}) and the default risk "
            f"({default}) are below the limits. Profile factors: {factors}.")

BAD_PHRASES = ["i can't", "i can’t", "i cannot", "i'm sorry", "i am sorry",
               "can't assist", "as an ai", "loan officer", "here is", "here are"]

def clean_llm_text(text):
    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    # drop a leading "Here is ...:" line if the model added one
    if lines and (lines[0].lower().startswith("here") or lines[0].endswith(":")):
        lines = lines[1:]
    return " ".join(lines).strip().strip('"')

def facts_text(result, customer_hc_data):
    """Exact facts, written by Python (never by the LLM)."""
    fraud = result["Agent 1 (RNN) Fraud Risk"]
    default = result["Agent 2 (RF) Default Risk"]
    reason = result["Reason"]
    factors = top_features_text(customer_hc_data)
    F, R = FRAUD_THRESHOLD * 100, RF_THRESHOLD * 100
    if reason.startswith("High fraud"):
        return (f"Fraud risk is {fraud}, above the {F:.0f}% limit; "
                f"default risk is {default}. Profile factors: {factors}.")
    elif reason.startswith("High default"):
        return (f"Default risk is {default}, above the {R:.0f}% limit; "
                f"fraud risk is {fraud}. Profile factors: {factors}.")
    return (f"Fraud risk is {fraud} (limit {F:.0f}%) and default risk is {default} "
            f"(limit {R:.0f}%). Profile factors: {factors}.")

def explain_with_llm(result, customer_hc_data):
    """The LLM writes ONE opening sentence (no numbers); Python adds the exact facts."""
    base = explain_decision(result, customer_hc_data)     # safe fallback
    approved = "APPROVED" in result["Final Decision"]
    reason = result["Reason"]
    verb = "approved" if approved else "rejected"
    prompt = f"""Write ONE short, professional sentence announcing that a loan application is {verb}.
The reason is: {reason.lower()}.
Do not use any digits or numbers. Do not mention any other fact.
Output only the sentence."""
    try:
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
                  "options": {"temperature": 0.4, "num_predict": 60}},
            timeout=300)
        r.raise_for_status()
        sentence = clean_llm_text(r.json()["response"])
        low = sentence.lower()

        ok = 20 <= len(sentence) <= 250
        ok = ok and not any(ch.isdigit() for ch in sentence)        # no numbers allowed
        ok = ok and not any(p in low for p in BAD_PHRASES)           # no refusal / preamble
        if approved:
            ok = ok and "approv" in low and "reject" not in low and "declin" not in low
        else:
            ok = (ok and "reject" in low and "is approved" not in low
                  and "been approved" not in low and "be approved" not in low)
            # the sentence must not give a wrong reason
            if reason.startswith("High default"):
                ok = ok and "fraud" not in low
            if reason.startswith("High fraud"):
                ok = ok and "default" not in low
        if ok:
            return f"{sentence} {facts_text(result, customer_hc_data)}", "LLM sentence + exact facts"
    except Exception:
        pass
    return base, "template (fallback)"

In [18]:
def agent3_decision(customer_cc_data, customer_hc_data):
    """Agent 3: rules take the decision, a local LLM rewrites the justification."""
    result = agent3_rules(customer_cc_data, customer_hc_data)
    text, source = explain_with_llm(result, customer_hc_data)
    result["Justification"] = text
    result["Justification source"] = source
    return result

print("✅ Agent 3 (decision rules + checked LLM justification) ready.")

✅ Agent 3 (decision rules + checked LLM justification) ready.


In [19]:
import time

# Check that Ollama is running
try:
    requests.get("http://localhost:11434", timeout=3)
    print("✅ Ollama is running: justifications come from the local LLM\n")
except Exception:
    print("⚠️ Ollama is NOT running: template fallback will be used\n")

with open("../data/clean/cc_data.pkl", "rb") as f:
    _, X_cc_test, _, y_cc_test = pickle.load(f)
with open("../data/clean/hc_data.pkl", "rb") as f:
    _, X_hc_test, _, y_hc_test, _ = pickle.load(f)

# Scores of every test row (used only to pick clear example cases)
rnn_probs = agent1_rnn.predict(
    X_cc_test.values.reshape(-1, 1, X_cc_test.shape[1]), verbose=0).flatten()
rf_probs = agent2_rf.predict_proba(X_hc_test)[:, 1]

normal_pos = int(np.where(y_cc_test.values == 0)[0][0])
fraud_positions = np.where(y_cc_test.values == 1)[0]
fraud_pos = int(fraud_positions[np.argmax(rnn_probs[fraud_positions])])
good_profile = int(np.argmin(rf_probs))
risky_profile = int(np.argmax(rf_probs))

cases = {
    "A: good profile + normal transactions":     (normal_pos, good_profile),
    "B: risky profile + normal transactions":    (normal_pos, risky_profile),
    "C: good profile + fraudulent transaction":  (fraud_pos, good_profile),
    "D: risky profile + fraudulent transaction": (fraud_pos, risky_profile),
}

for name, (i, j) in cases.items():
    t0 = time.time()
    r = agent3_decision(X_cc_test.values[i], X_hc_test[j])
    print(f"=== {name}  ({time.time()-t0:.0f}s) ===")
    for k, v in r.items():
        print(f"{k}: {v}")
    print()

✅ Ollama is running: justifications come from the local LLM

=== A: good profile + normal transactions  (4s) ===
Agent 1 (RNN) Fraud Risk: 0.00%
Agent 2 (RF) Default Risk: 6.94%
Final Decision: APPROVED ✅
Reason: Low fraud risk and low default risk
Justification: The loan application has been approved due to its low risk profile. Fraud risk is 0.00% (limit 50%) and default risk is 6.94% (limit 50%). Profile factors: external credit score 3 is close to average; external credit score 2 is higher than average (favorable); employment length is longer than average (favorable).
Justification source: LLM sentence + exact facts

=== B: risky profile + normal transactions  (4s) ===
Agent 1 (RNN) Fraud Risk: 0.00%
Agent 2 (RF) Default Risk: 82.52%
Final Decision: REJECTED ❌
Reason: High default risk based on profile
Justification: The loan application has been rejected due to concerns regarding the applicant's creditworthiness. Default risk is 82.52%, above the 50% limit; fraud risk is 0.00%. Pr